# Stateful safe-admission proof

This notebook demonstrates the intended HRL separation using the real `SafeAdmissionController`:

1. The upper bias requests a release volume.
2. A negative directional A3 offset creates radio-eligible candidates.
3. Safe admission ranks those candidates and permits only the window quota.
4. The quota remains exhausted across later local steps in the same upper window.
5. A new upper PPO step recomputes the quota from the new bias and network state.

The central proof case has three boundary UEs. All three satisfy A3, but only two are admitted.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from safe_admission_controller import SafeAdmissionController

print(f"Repository root: {ROOT}")

## Helper: A3 creates candidates, not completed handovers

A UE is eligible when

$$RSRP_{target} > RSRP_{serving} + O_{i,j,s} + H_{A3}.$$

The helper below applies that condition and builds the candidate records consumed by safe admission.

In [ ]:
def build_a3_candidates(ues, *, offset_db, hysteresis_db, target_load=0.40):
    candidates = []
    for ue in ues:
        threshold = ue["rsrp_serving_dbm"] + offset_db + hysteresis_db
        eligible = ue["rsrp_target_dbm"] > threshold
        margin = ue["rsrp_target_dbm"] - threshold
        if eligible:
            candidates.append({
                "ue_id": ue["ue_id"],
                "source_id": 0,
                "target_id": 1,
                "slice_type": "eMBB",
                "a3_margin": margin,
                "radio_delta_db": (
                    ue["rsrp_target_dbm"] - ue["rsrp_serving_dbm"]
                ),
                "target_load": target_load,
                "target_load_increment": ue["load_contribution"],
                "source_load_contribution": ue["load_contribution"],
                "target_safe_limit": 0.80,
                "target_sla_severity": ue.get("target_sla_severity", 0.0),
                "handover_failure_ratio": ue.get("failure_ratio", 0.0),
                "pingpong_ratio": ue.get("pingpong_ratio", 0.0),
            })
    return candidates


boundary_ues = [
    {"ue_id": 101, "rsrp_serving_dbm": -80.0, "rsrp_target_dbm": -82.0, "load_contribution": 0.04},
    {"ue_id": 102, "rsrp_serving_dbm": -80.0, "rsrp_target_dbm": -83.5, "load_contribution": 0.04},
    {"ue_id": 103, "rsrp_serving_dbm": -80.0, "rsrp_target_dbm": -85.5, "load_contribution": 0.04},
]

offset_db = -8.0
hysteresis_db = 1.0
candidates = build_a3_candidates(
    boundary_ues,
    offset_db=offset_db,
    hysteresis_db=hysteresis_db,
)

print(f"Negative offset: {offset_db:.1f} dB")
print(f"A3-eligible UEs: {[c['ue_id'] for c in candidates]}")
print(f"A3 margins: {[round(c['a3_margin'], 2) for c in candidates]}")
assert len(candidates) == 3, "The proof requires all three UEs to satisfy A3."


## Upper window 1: all three are eligible, quota allows only two

For eMBB loads `[0.90, 0.40, 0.50]`, the balance target is `0.60`. With bias `-0.5`:

- source excess = `0.90 - 0.60 = 0.30`;
- requested release load = `0.5 × 0.30 = 0.15`;
- estimated load per source UE = `0.90 / 10 = 0.09`;
- UE quota = `ceil(0.15 / 0.09) = 2`.

In [ ]:
controller = SafeAdmissionController(bias_deadband=0.05)

loads_window_1 = {
    (0, "eMBB"): 0.90,
    (1, "eMBB"): 0.40,
    (2, "eMBB"): 0.50,
}
ue_counts_window_1 = {
    (0, "eMBB"): 10,
    (1, "eMBB"): 4,
    (2, "eMBB"): 5,
}
bias_window_1 = np.array([[-0.5], [0.0], [0.0]], dtype=np.float32)

quota_1 = controller.begin_upper_window(
    bias_matrix=bias_window_1,
    gnb_ids=[0, 1, 2],
    slice_types=["eMBB"],
    loads=loads_window_1,
    ue_counts=ue_counts_window_1,
    remaining_handover_budget=20,
)

state_before = controller.get_state()
print("Window:", state_before["window_id"])
print("Source excess load:", round(state_before["source_excess_load"][(0, "eMBB")], 3))
print("Requested release load:", round(state_before["requested_release_load"][(0, "eMBB")], 3))
print("Release quota:", quota_1[(0, "eMBB")])

accepted_1, rejected_1, debug_1 = controller.admit_candidates(
    candidates,
    max_acceptances=10,
    remaining_handover_budget=20,
    current_tick=1,
)

print("Accepted UEs:", [item["ue_id"] for item in accepted_1])
print("Rejected:", [(item["ue_id"], item["rejection_reason"]) for item in rejected_1])

assert quota_1[(0, "eMBB")] == 2
assert len(accepted_1) == 2
assert len(rejected_1) == 1
assert rejected_1[0]["rejection_reason"] == "quota_exhausted"

# The simulator calls commit only after each selected handover succeeds.
for decision in accepted_1:
    controller.commit(decision)

state_after = controller.get_state()
print("Used quota after successful HOs:", state_after["used"][(0, "eMBB")])
print("Remaining quota:", state_after["remaining"][(0, "eMBB")])
assert state_after["used"][(0, "eMBB")] == 2
assert state_after["remaining"][(0, "eMBB")] == 0


## Same upper window: repeated A3 eligibility cannot bypass the quota

UE 103 still satisfies the negative-offset A3 condition on the next local step. It must remain attached because the source-slice quota is already exhausted.

In [ ]:
remaining_candidate = [c for c in candidates if c["ue_id"] == 103]
accepted_repeat, rejected_repeat, debug_repeat = controller.admit_candidates(
    remaining_candidate,
    max_acceptances=10,
    remaining_handover_budget=18,
    current_tick=2,
)

print("Accepted on later local step:", accepted_repeat)
print("Rejected on later local step:", [
    (item["ue_id"], item["rejection_reason"]) for item in rejected_repeat
])

assert accepted_repeat == []
assert rejected_repeat[0]["ue_id"] == 103
assert rejected_repeat[0]["rejection_reason"] == "quota_exhausted"


## Upper window 2: new state recomputes the quota

At the next PPO step, the previous `used=2` counter is discarded. New loads `[0.72, 0.54, 0.54]` and eight remaining source UEs produce a new quota of one UE.

In [ ]:
loads_window_2 = {
    (0, "eMBB"): 0.72,
    (1, "eMBB"): 0.54,
    (2, "eMBB"): 0.54,
}
ue_counts_window_2 = {
    (0, "eMBB"): 8,
    (1, "eMBB"): 6,
    (2, "eMBB"): 5,
}

quota_2 = controller.begin_upper_window(
    bias_matrix=bias_window_1,
    gnb_ids=[0, 1, 2],
    slice_types=["eMBB"],
    loads=loads_window_2,
    ue_counts=ue_counts_window_2,
    remaining_handover_budget=18,
)
state_window_2 = controller.get_state()

print("New window:", state_window_2["window_id"])
print("Used counter after reset:", state_window_2["used"][(0, "eMBB")])
print("Recomputed quota:", quota_2[(0, "eMBB")])

window_2_candidate = dict(remaining_candidate[0])
window_2_candidate["target_load"] = 0.54
accepted_2, rejected_2, _ = controller.admit_candidates([window_2_candidate])

print("Accepted in new window:", [item["ue_id"] for item in accepted_2])
assert state_window_2["used"][(0, "eMBB")] == 0
assert quota_2[(0, "eMBB")] == 1
assert [item["ue_id"] for item in accepted_2] == [103]


## Target safety remains a hard veto

Even with release pressure and available source quota, a candidate is rejected if its estimated target load would exceed the configured safe limit.

In [ ]:
unsafe_controller = SafeAdmissionController()
unsafe_controller.begin_upper_window(
    bias_matrix=np.array([[-1.0], [0.0], [0.0]], dtype=np.float32),
    gnb_ids=[0, 1, 2],
    slice_types=["eMBB"],
    loads={(0, "eMBB"): 0.90, (1, "eMBB"): 0.79, (2, "eMBB"): 0.40},
    ue_counts={(0, "eMBB"): 10, (1, "eMBB"): 8, (2, "eMBB"): 4},
)

unsafe_candidate = dict(candidates[0])
unsafe_candidate["target_load"] = 0.79
unsafe_candidate["target_load_increment"] = 0.04
unsafe_candidate["target_safe_limit"] = 0.80

accepted_unsafe, rejected_unsafe, _ = unsafe_controller.admit_candidates(
    [unsafe_candidate]
)
print("Unsafe target accepted:", accepted_unsafe)
print("Reason:", rejected_unsafe[0]["rejection_reason"])

assert accepted_unsafe == []
assert rejected_unsafe[0]["rejection_reason"] == "target_safety"


## Visual summary

In [ ]:
labels = ["A3 eligible\nwindow 1", "Admitted\nwindow 1", "Later local step", "New upper\nwindow"]
values = [3, 2, 0, 1]
colors = ["#4C78A8", "#59A14F", "#E15759", "#F28E2B"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values, color=colors)
ax.axhline(2, color="black", linestyle="--", linewidth=1, label="window-1 quota = 2")
ax.set_ylabel("Number of UEs")
ax.set_title("A3 eligibility is separated from admitted migration volume")
ax.set_ylim(0, 3.6)
ax.legend()
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.08, str(value), ha="center")
plt.tight_layout()
plt.show()


## Result

The mechanism is proven when the final cell prints `PASS`:

- strong negative offset created all three candidates;
- only two candidates were admitted in upper window 1;
- the remaining eligible UE was denied during later local steps;
- upper window 2 reset and recomputed the quota;
- target safety independently vetoed an unsafe handover.

In [ ]:
assert len(candidates) == 3
assert len(accepted_1) == 2
assert accepted_repeat == []
assert len(accepted_2) == 1
assert accepted_unsafe == []

print("PASS: A3 generated candidates, while stateful safe admission controlled migration volume.")